# AgenticForecaster — Evaluation Notebook

This notebook benchmarks `AgenticForecaster` against naïve baselines on two classic time-series datasets.

It also illustrates the key difference between:
- **Mock backend** — deterministic rule-based policy (no API key needed)
- **LLM backend** — reads the prompt and reasons about the data (Gemini / OpenAI / Claude)

```
pip install -e ".[sktime]"   # to run this notebook
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sktime.datasets import load_airline, load_shampoo_sales
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.trend import PolynomialTrendForecaster
from sktime.forecasting.exp_smoothing import ExponentialSmoothing
from sktime.performance_metrics.forecasting import mean_absolute_percentage_error

from sktime_agentic import AgenticForecaster
from sktime_agentic.tools import summarize_data

## Dataset 1 — Airline passengers (seasonal + trend)

Monthly international airline passengers 1949–1960.  
Strong multiplicative trend and yearly seasonality (sp=12).

In [ ]:
y_airline = load_airline()
holdout = 12
train_a, test_a = y_airline.iloc[:-holdout], y_airline.iloc[-holdout:]
fh_a = list(range(1, holdout + 1))

# Data fingerprint
fp = summarize_data(train_a)
print('Fingerprint:')
for k, v in fp.items():
    print(f'  {k}: {v}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
train_a.plot(ax=ax, label='Train', color='steelblue')
test_a.plot(ax=ax, label='Test (holdout)', color='darkorange', linestyle='--')
ax.set_title('Airline passengers — monthly')
ax.set_ylabel('Passengers (thousands)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Baselines
baselines_a = {
    'NaiveForecaster (last)':   NaiveForecaster(strategy='last'),
    'NaiveForecaster (mean)':   NaiveForecaster(strategy='mean'),
    'SeasonalNaive (sp=12)':    NaiveForecaster(strategy='last', sp=12),
    'ExponentialSmoothing':     ExponentialSmoothing(trend='add', seasonal='add', sp=12),
}

results_a = {}
for name, model in baselines_a.items():
    model.fit(train_a)
    mape = mean_absolute_percentage_error(test_a, model.predict(fh_a))
    results_a[name] = mape
    print(f'{name:<35} MAPE = {mape:.4f}')

In [ ]:
# AgenticForecaster — mock backend
af_airline = AgenticForecaster(
    prompt='Monthly airline passengers. Strong yearly seasonality and upward trend.',
    backend='mock',
    holdout=holdout,
)
af_airline.fit(train_a, fh=fh_a)
af_mape_a = mean_absolute_percentage_error(test_a, af_airline.predict())
results_a['AgenticForecaster (mock)'] = af_mape_a

print(f'Selected : {af_airline.selected_}  params={af_airline.selected_params_}')
print(f'Rationale: {af_airline.rationale_}')
print(f'MAPE     : {af_mape_a:.4f}')

In [ ]:
# Forecast comparison plot
fig, ax = plt.subplots(figsize=(11, 4))
y_airline.iloc[-24:].plot(ax=ax, color='black', label='Actual', linewidth=1.5)

colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
for (name, model), color in zip(baselines_a.items(), colors):
    pred = model.predict(fh_a)
    pred.plot(ax=ax, color=color, linestyle='--', alpha=0.7,
              label=f'{name} ({results_a[name]:.3f})')

af_airline.predict().plot(ax=ax, color='black', linestyle='-.',
                          linewidth=2, label=f'AgenticForecaster mock ({af_mape_a:.3f})')

ax.axvline(x=test_a.index[0], color='gray', linestyle=':', alpha=0.5, label='Train/test split')
ax.set_title('Airline — forecast comparison (MAPE in legend)')
ax.set_ylabel('Passengers (thousands)')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

## Dataset 2 — Shampoo sales (trend, no seasonality)

Monthly shampoo sales over 3 years.  
Strong upward trend, no meaningful seasonality.

This dataset exposes the **key limitation of the mock policy**: because it only evaluates
naive forecasters, it misses the trend. A real LLM reads the prompt and picks `PolynomialTrendForecaster`.

In [ ]:
y_shampoo = load_shampoo_sales()
holdout_s = 6
train_s, test_s = y_shampoo.iloc[:-holdout_s], y_shampoo.iloc[-holdout_s:]
fh_s = list(range(1, holdout_s + 1))

fp_s = summarize_data(train_s)
print('Fingerprint:')
for k, v in fp_s.items():
    print(f'  {k}: {v}')

In [ ]:
baselines_s = {
    'NaiveForecaster (last)':    NaiveForecaster(strategy='last'),
    'NaiveForecaster (mean)':    NaiveForecaster(strategy='mean'),
    'PolynomialTrendForecaster':  PolynomialTrendForecaster(degree=1),
    'ExponentialSmoothing':      ExponentialSmoothing(trend='add'),
}

results_s = {}
for name, model in baselines_s.items():
    model.fit(train_s)
    mape = mean_absolute_percentage_error(test_s, model.predict(fh_s))
    results_s[name] = mape
    print(f'{name:<35} MAPE = {mape:.4f}')

In [ ]:
# Mock backend — limited to naive candidates
af_shampoo_mock = AgenticForecaster(
    prompt='Monthly shampoo sales with strong upward trend, no seasonality.',
    backend='mock',
    holdout=holdout_s,
)
af_shampoo_mock.fit(train_s, fh=fh_s)
af_mape_mock = mean_absolute_percentage_error(test_s, af_shampoo_mock.predict())
results_s['AgenticForecaster (mock)'] = af_mape_mock

print(f'Mock selected : {af_shampoo_mock.selected_}  params={af_shampoo_mock.selected_params_}')
print(f'Mock MAPE     : {af_mape_mock:.4f}')
print()
print('Note: mock policy ignores prompt text — it only uses ACF-based seasonality detection.')
print('A real LLM reads "strong upward trend" and picks PolynomialTrendForecaster.')

In [ ]:
# Summary table
print('\n=== Airline (seasonal + trend) ===')
for name, mape in sorted(results_a.items(), key=lambda x: x[1]):
    marker = ' ◄ agentic' if 'Agentic' in name else ''
    print(f'  {name:<35} {mape:.4f}{marker}')

print('\n=== Shampoo (trend only) ===')
for name, mape in sorted(results_s.items(), key=lambda x: x[1]):
    marker = ' ◄ agentic' if 'Agentic' in name else ''
    print(f'  {name:<35} {mape:.4f}{marker}')

## Takeaways

| Dataset | Pattern | Mock result | LLM advantage |
|---|---|---|---|
| Airline | Seasonal + trend | ✅ Picks SeasonalNaive(sp=12) — competitive | Marginal: mock policy already handles seasonality well |
| Shampoo | Trend only | ❌ Picks SeasonalNaive(sp=4) — spurious ACF peak | LLM reads prompt, picks PolynomialTrendForecaster |

**The mock policy** is a fast, offline stand-in. It works well when seasonality dominates 
because the ACF-based detector is reliable for strong periodic signals.

**A real LLM backend** adds genuine value when:
- The user prompt contains information the data fingerprint can't capture ("no seasonality", "sudden level shift", "holiday effects")
- The best model requires non-naive candidates (trend forecasters, ARIMA, ETS)
- The series is short and ACF estimates are noisy

This is the core motivation for the agentic design: **the LLM reads both the data fingerprint 
and the English prompt**, combining quantitative evidence with domain knowledge.